In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("data.csv", inferSchema=True, header=True)
df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 16:07:08 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlan0)
26/08/24 16:07:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/jcalvarezj/Documentos/repos/practice/pyspark-example/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 16:07:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+-------+-----+
|region|product|sales|
+------+-------+-----+
| North|      A| 1200|
| North|      B|  800|
| North|      C|  400|
| South|      A| 1000|
| South|      B|  500|
| South|      C|  500|
+------+-------+-----+



In [2]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

region_window = Window.partitionBy("region")
ranking_window = Window.partitionBy("region").orderBy(F.desc("sales"))

df_sales_by_region = df.withColumns({
    "total_region": F.sum("sales").over(region_window),
    "ranking": F.rank().over(ranking_window)
}).withColumn("pct_region", F.round(100 * F.col("sales") / F.col("total_region"), 2) )

df_sales_by_region.show()

+------+-------+-----+------------+-------+----------+
|region|product|sales|total_region|ranking|pct_region|
+------+-------+-----+------------+-------+----------+
| North|      A| 1200|        2400|      1|      50.0|
| North|      B|  800|        2400|      2|     33.33|
| North|      C|  400|        2400|      3|     16.67|
| South|      A| 1000|        2000|      1|      50.0|
| South|      B|  500|        2000|      2|      25.0|
| South|      C|  500|        2000|      2|      25.0|
+------+-------+-----+------------+-------+----------+

